# Phase 5C - Secondary metal dissolution

Small regression models for Li, Ga and Sc extraction efficiency from bauxite residue, trained on the lab experiment data.

## 1. Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict, LeaveOneOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
import sklearn; print(f"Scikit-learn: {sklearn.__version__}")

SEED = 42
np.random.seed(SEED)

In [ ]:
# Upload the CSV files
from google.colab import files
print("Upload: li_dissolution_clean.csv, ga_dissolution_clean.csv, sc_dissolution_clean.csv")
uploaded = files.upload()

## 2. Data Loading & Inspection

In [ ]:
# Load all three datasets
li_raw = pd.read_csv('li_dissolution_clean.csv')
ga_raw = pd.read_csv('ga_dissolution_clean.csv')
sc_raw = pd.read_csv('sc_dissolution_clean.csv')

print(f"Li: {li_raw.shape}")
print(f"Ga: {ga_raw.shape}")
print(f"Sc: {sc_raw.shape}")

print("\n" + "="*60)
print("Li dissolution columns:")
print(list(li_raw.columns))

In [ ]:
# Display first few rows
print("Li dissolution preview:")
li_raw.head()

## 3. Data cleaning and features

Each element sheet has three experiment blocks (roasting, lime addition, temperature). These get parsed into one table.

In [ ]:
def parse_dissolution_data(df, element_name):
    """
    Parse the multi-block dissolution sheet into a clean tabular format.
    
    The sheets have 3 experimental blocks side-by-side:
    - Block 1: Roasting conditions (sample_id, condition/time)
    - Block 2: Lime addition effects
    - Block 3: Temperature effects
    """
    records = []
    
    # Find target column (extraction efficiency)
    eff_cols = [c for c in df.columns if 'Extraction efficiency' in c]
    print(f"\n{element_name} efficiency columns: {eff_cols}")
    
    # Block 1: Roasting (first efficiency column)
    if len(eff_cols) >= 1:
        block1 = df[df.iloc[:, 0].notna() & (df.iloc[:, 0] != 'NaN')].copy()
        for _, row in block1.iterrows():
            records.append({
                'element': element_name,
                'sample_id': str(row.iloc[0]).strip(),
                'experimental_block': 'roasting',
                'bauxite_concentration': pd.to_numeric(row.iloc[1], errors='coerce'),
                'condition': str(row.iloc[3]) if len(row) > 3 else None,
                'extraction_efficiency_pct': pd.to_numeric(row[eff_cols[0]], errors='coerce')
            })
    
    # Block 2: Lime addition (second efficiency column)
    if len(eff_cols) >= 2:
        lime_col = [c for c in df.columns if 'Lime' in c]
        if lime_col:
            block2 = df[df[lime_col[0]].notna()].copy()
            for _, row in block2.iterrows():
                lime_val = pd.to_numeric(row[lime_col[0]], errors='coerce')
                if pd.notna(lime_val):
                    records.append({
                        'element': element_name,
                        'sample_id': None,
                        'experimental_block': 'lime',
                        'bauxite_concentration': pd.to_numeric(row.iloc[10] if len(row) > 10 else None, errors='coerce'),
                        'lime_pct': lime_val,
                        'extraction_efficiency_pct': pd.to_numeric(row[eff_cols[1]], errors='coerce')
                    })
    
    # Block 3: Temperature (third efficiency column)
    if len(eff_cols) >= 3:
        temp_col = [c for c in df.columns if 'temperature' in c.lower()]
        if temp_col:
            block3 = df[df[temp_col[0]].notna()].copy()
            for _, row in block3.iterrows():
                temp_val = pd.to_numeric(row[temp_col[0]], errors='coerce')
                if pd.notna(temp_val):
                    records.append({
                        'element': element_name,
                        'sample_id': None,
                        'experimental_block': 'temperature',
                        'bauxite_concentration': pd.to_numeric(row.iloc[19] if len(row) > 19 else None, errors='coerce'),
                        'temperature_C': temp_val,
                        'extraction_efficiency_pct': pd.to_numeric(row[eff_cols[2]], errors='coerce')
                    })
    
    clean_df = pd.DataFrame(records)
    clean_df = clean_df[clean_df['extraction_efficiency_pct'].notna()]
    return clean_df


li_clean = parse_dissolution_data(li_raw, 'Li')
ga_clean = parse_dissolution_data(ga_raw, 'Ga')
sc_clean = parse_dissolution_data(sc_raw, 'Sc')

print(f"\nCleaned shapes:")
print(f"  Li: {li_clean.shape}")
print(f"  Ga: {ga_clean.shape}")
print(f"  Sc: {sc_clean.shape}")

In [ ]:
# Inspect cleaned Li data
print("Li dissolution cleaned:")
print(li_clean.head(10))
print(f"\nLi target stats:")
print(li_clean['extraction_efficiency_pct'].describe())

## 4. Exploratory Data Analysis

In [ ]:
# Target distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (df, name, color) in enumerate([
    (li_clean, 'Li', 'tab:blue'),
    (ga_clean, 'Ga', 'tab:orange'),
    (sc_clean, 'Sc', 'tab:green')
]):
    axes[i].hist(df['extraction_efficiency_pct'], bins=10, color=color, edgecolor='black', alpha=0.7)
    axes[i].axvline(df['extraction_efficiency_pct'].mean(), color='red', linestyle='--', label='Mean')
    axes[i].set_xlabel('Extraction Efficiency (%)')
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'{name} Dissolution Efficiency (n={len(df)})')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Efficiency by experimental block
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (df, name) in enumerate([(li_clean, 'Li'), (ga_clean, 'Ga'), (sc_clean, 'Sc')]):
    df.boxplot(column='extraction_efficiency_pct', by='experimental_block', ax=axes[i])
    axes[i].set_title(f'{name} by Block')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Extraction Efficiency (%)')
    plt.suptitle('')

plt.tight_layout()
plt.show()

## 5. Model training

Only 23-31 rows per element, so this uses leave-one-out CV with Ridge and a shallow random forest, compared against a mean baseline.

In [ ]:
def prepare_features(df):
    """Convert cleaned dissolution dataframe to feature matrix."""
    feat = pd.DataFrame(index=df.index)
    
    # Bauxite concentration
    feat['bauxite_conc'] = df['bauxite_concentration'].fillna(df['bauxite_concentration'].mean())
    
    # Experimental block indicators
    feat['is_roasting'] = (df['experimental_block'] == 'roasting').astype(int)
    feat['is_lime'] = (df['experimental_block'] == 'lime').astype(int)
    feat['is_temperature'] = (df['experimental_block'] == 'temperature').astype(int)
    
    # Factor values (fill NaN with 0)
    feat['lime_pct'] = df.get('lime_pct', 0).fillna(0)
    feat['temperature_C'] = df.get('temperature_C', 0).fillna(0)
    
    return feat


def train_element_model(df, element_name):
    """Train dissolution model for one element."""
    X = prepare_features(df).values
    y = df['extraction_efficiency_pct'].values
    
    print(f"\n{'='*60}")
    print(f"{element_name} Dissolution Model")
    print(f"{'='*60}")
    print(f"n = {len(y)}")
    print(f"Features: {prepare_features(df).columns.tolist()}")
    
    # Mean baseline
    y_mean = np.full_like(y, y.mean())
    baseline_mae = mean_absolute_error(y, y_mean)
    baseline_rmse = np.sqrt(mean_squared_error(y, y_mean))
    
    # Ridge
    ridge = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=5.0, random_state=SEED))
    ])
    loo = LeaveOneOut()
    y_pred_ridge = cross_val_predict(ridge, X, y, cv=loo)
    ridge_mae = mean_absolute_error(y, y_pred_ridge)
    ridge_rmse = np.sqrt(mean_squared_error(y, y_pred_ridge))
    ridge_r2 = r2_score(y, y_pred_ridge)
    
    # Random Forest
    rf = RandomForestRegressor(n_estimators=50, max_depth=3, min_samples_leaf=2, random_state=SEED)
    y_pred_rf = cross_val_predict(rf, X, y, cv=loo)
    rf_mae = mean_absolute_error(y, y_pred_rf)
    rf_rmse = np.sqrt(mean_squared_error(y, y_pred_rf))
    rf_r2 = r2_score(y, y_pred_rf)
    
    results = pd.DataFrame([
        {'model': 'Mean Baseline', 'MAE': round(baseline_mae, 2), 'RMSE': round(baseline_rmse, 2), 'R2': None},
        {'model': 'Ridge (α=5)', 'MAE': round(ridge_mae, 2), 'RMSE': round(ridge_rmse, 2), 'R2': round(ridge_r2, 3)},
        {'model': 'Random Forest', 'MAE': round(rf_mae, 2), 'RMSE': round(rf_rmse, 2), 'R2': round(rf_r2, 3)},
    ])
    
    print("\nLOOCV Results:")
    print(results.to_string(index=False))
    
    # Train final model on all data
    ridge.fit(X, y)
    
    return {
        'element': element_name,
        'model': ridge,
        'n': len(y),
        'feature_names': prepare_features(df).columns.tolist(),
        'results': results,
        'y_pred': y_pred_ridge,
        'y_true': y,
    }

In [ ]:
# Train all three models
li_model = train_element_model(li_clean, 'Li')
ga_model = train_element_model(ga_clean, 'Ga')
sc_model = train_element_model(sc_clean, 'Sc')

## 6. Model Visualization

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, model_dict in enumerate([li_model, ga_model, sc_model]):
    y_true = model_dict['y_true']
    y_pred = model_dict['y_pred']
    name = model_dict['element']
    
    axes[i].scatter(y_true, y_pred, alpha=0.7, s=60)
    axes[i].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'k--', lw=1)
    axes[i].set_xlabel('Actual Extraction Efficiency (%)')
    axes[i].set_ylabel('Predicted Extraction Efficiency (%)')
    axes[i].set_title(f'{name} — Ridge LOOCV (n={model_dict["n"]})')
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, model_dict in enumerate([li_model, ga_model, sc_model]):
    y_true = model_dict['y_true']
    y_pred = model_dict['y_pred']
    residuals = y_true - y_pred
    name = model_dict['element']
    
    axes[i].scatter(y_pred, residuals, alpha=0.7, s=60)
    axes[i].axhline(0, color='k', linestyle='--', lw=1)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Residual')
    axes[i].set_title(f'{name} Residuals')
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Export Models

In [ ]:
import pickle

# Package all three models
secondary_models = {
    'Li': {
        'model': li_model['model'],
        'feature_names': li_model['feature_names'],
        'n_train': li_model['n'],
        'loocv_results': li_model['results'],
    },
    'Ga': {
        'model': ga_model['model'],
        'feature_names': ga_model['feature_names'],
        'n_train': ga_model['n'],
        'loocv_results': ga_model['results'],
    },
    'Sc': {
        'model': sc_model['model'],
        'feature_names': sc_model['feature_names'],
        'n_train': sc_model['n'],
        'loocv_results': sc_model['results'],
    },
    'metadata': {
        'scope': 'proof-of-concept, laboratory data only',
        'target': 'extraction_efficiency_pct',
        'seed': SEED,
        'note': 'Predicts Li/Ga/Sc recovery from bauxite residue for secondary product recommendation'
    }
}

with open('secondary_metal_dissolution_models.pkl', 'wb') as f:
    pickle.dump(secondary_models, f)

print("Models saved to secondary_metal_dissolution_models.pkl")

# Download
from google.colab import files
files.download('secondary_metal_dissolution_models.pkl')

In [ ]:
# Save combined results
all_results = pd.concat([
    li_model['results'].assign(element='Li'),
    ga_model['results'].assign(element='Ga'),
    sc_model['results'].assign(element='Sc'),
], ignore_index=True)

all_results.to_csv('phase5c_secondary_metals_results.csv', index=False)
print("Results saved to phase5c_secondary_metals_results.csv")
files.download('phase5c_secondary_metals_results.csv')

## 8. Summary

| Element | n |
|---|---|
| Li | 31 |
| Ga | 23 |
| Sc | 23 |

LOOCV scores are in the results above. Lab scale only.